
PCL-PUF and XOR-PUF Attack Notebook
=================================
This file provides attack scripts for both:
- the PCL-PUF linear approximation attack, and
- a baseline XOR-PUF logistic-regression attack.

The notebook is designed to reproduce the attack workflow described in the paper.
You can change the configuration parameters (for example, the number of APUF
components, dataset size, and training/test sizes) to regenerate the reported
results under different settings.

Requirements:
    pip install numpy scipy scikit-learn
"""

In [ ]:



import numpy as np
from scipy import sparse
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import accuracy_score

# ──────────────────────────────────────────────────────────────────────────────
#  1.  REAP-NVM PUF SIMULATION
# ──────────────────────────────────────────────────────────────────────────────

def ReapNVM(num_bits, seed, sigma_proc=0.05):
    rng = np.random.default_rng(seed)
    chal_length = num_bits
    n_levels = 4

    R_levels_nom = np.array([10e3, 75e3, 125e3, 275e3])
    log_R_levels_nom = np.log10(R_levels_nom)
    tR = np.zeros((2, n_levels, chal_length))

    for row in range(2):
        for stage in range(chal_length):
            log_levels = log_R_levels_nom + sigma_proc * rng.standard_normal(n_levels)
            levels = 10 ** log_levels
            tR[row, :, stage] = levels * 250e-12

    tSW = np.full((2, 2, chal_length), 372.0 / 1e12)
    return 4.0 * tR, 4.0 * tSW

def ReapNVM_evaluate(PUF, challenge, position, value, chunk_size=100_000):
    chal      = (challenge + 1) / 2.0
    tR4, tSW4 = PUF
    chalpos   = position.astype(int).flatten()
    chalval   = value.astype(int).flatten()
    N         = chal.shape[0]
    responses = np.zeros(N, dtype=np.int8)
    for start in range(0, N, chunk_size):
        end = min(start + chunk_size, N)
        n   = end - start
        cc  = chal[start:end]
        pc  = chalpos[start:end]
        vc  = chalval[start:end]
        tv1 = np.tile(tR4[0, 0, :], (n, 1)); tv2 = np.tile(tR4[1, 0, :], (n, 1))
        tv1[np.arange(n), pc] = tR4[0, vc, pc]
        tv2[np.arange(n), pc] = tR4[1, vc, pc]
        c   = np.bitwise_xor.accumulate(cc.astype(np.uint8), axis=1)
        t1 = np.sum(np.where(c == 0, tv1 + tSW4[0, 0, :], tv2 + tSW4[1, 0, :]), axis=1)
        t2 = np.sum(np.where(c == 0, tv2 + tSW4[0, 1, :], tv1 + tSW4[1, 1, :]), axis=1)
        responses[start:end] = (t1 > t2).astype(np.int8)
    return responses

# ──────────────────────────────────────────────────────────────────────────────
#  2.  APUF & OBFUSCATION LOGIC
# ──────────────────────────────────────────────────────────────────────────────

def apuf_generate(k, chal_size, seed=0):
    return np.random.default_rng(seed).normal(0, 1, (k, chal_size + 1))

def apuf_response(w, Phi):
    return (Phi @ w <= 0).astype(np.int8)

def dec_to_bin_vec(x, bitlen):
    return np.array([(x >> i) & 1 for i in range(bitlen)][::-1], dtype=np.uint8)

def bin_vec_to_dec(bits):
    out = 0
    for b in bits: out = (out << 1) | int(b)
    return out

def sliding_window_xor(bits, window_bits):
    window_bits = window_bits.astype(bits.dtype)
    for start in range(0, bits.shape[0], window_bits.shape[0]):
        end = min(start + window_bits.shape[0], bits.shape[0])
        bits[start:end] ^= window_bits[:end - start]
    return bits

def xor_obfuscate_position_value(position, value, upper_resp):
    N = position.shape[0]
    pos_out = np.zeros(N, dtype=np.uint32)
    val_out = np.zeros(N, dtype=np.uint32)
    for n in range(N):
        window_bits = upper_resp[:, n]
        pos_bits    = sliding_window_xor(dec_to_bin_vec(position[n], 7), window_bits)
        val_bits    = sliding_window_xor(dec_to_bin_vec(value[n],    2), window_bits)
        pos_out[n]  = bin_vec_to_dec(pos_bits)
        val_out[n]  = bin_vec_to_dec(val_bits)
    return pos_out, val_out

def pclpuf_evaluate(upper_w, lower_pufs, challenges, position, value):
    K_UP = upper_w.shape[0]
    N    = challenges.shape[0]
    Phi  = transform(challenges)
    upper_resp = np.array([apuf_response(upper_w[i], Phi) for i in range(K_UP)])
    pos_eff, val_eff = xor_obfuscate_position_value(position, value, upper_resp)
    xor_resp = np.zeros(N, dtype=np.int8)
    for puf in lower_pufs:
        xor_resp = np.bitwise_xor(xor_resp, ReapNVM_evaluate(puf, challenges, pos_eff, val_eff))
    return xor_resp

# ──────────────────────────────────────────────────────────────────────────────
#  3.  FEATURE ENGINEERING & PARITY TRANSFORMS
# ──────────────────────────────────────────────────────────────────────────────

def transform(challenges):
    N = challenges.shape[0]
    return np.hstack([np.cumprod(challenges, axis=1), np.ones((N, 1))])

def prepare_lr_features(Phi, positions, values, chal_size, n_levels):
    N         = Phi.shape[0]
    pos       = positions.astype(int)
    val       = values.astype(int)
    delta_idx = pos * n_levels + val
    data      = Phi[np.arange(N), pos]
    X_delta   = sparse.csr_matrix((data, (np.arange(N), delta_idx)),
                                   shape=(N, chal_size * n_levels))
    return sparse.hstack([sparse.csr_matrix(Phi), X_delta], format='csr')

# ──────────────────────────────────────────────────────────────────────────────
#  4.  CUSTOM XOR LR MATHEMATICAL SOLVER
# ──────────────────────────────────────────────────────────────────────────────

def xor_lr_loss_and_grad(w_flat, X_list, y, K, feat_size, lam=1e-4):
    N  = X_list[0].shape[0]
    w  = w_flat.reshape(K, feat_size)
    margins   = np.array([X_list[k].dot(w[k]) for k in range(K)])
    signs     = np.sign(margins)
    log_abs   = np.log(np.abs(margins) + 1e-12)
    prod_sign = np.prod(signs, axis=0)
    log_sum   = np.sum(log_abs, axis=0)
    product   = prod_sign * np.exp(np.clip(log_sum, -500, 500))
    p         = np.clip(expit(product), 1e-12, 1 - 1e-12)
    loss      = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    loss     += lam * 0.5 * np.dot(w_flat, w_flat)
    err       = (p - y) / N
    grad      = np.zeros_like(w)
    for k in range(K):
        others  = np.prod(margins[np.arange(K) != k], axis=0)
        grad[k] = X_list[k].T.dot(err * others) + lam * w[k]
    return loss, grad.flatten()

def predict_xor_lr(w_flat, X_list, K, feat_size):
    w         = w_flat.reshape(K, feat_size)
    margins   = np.array([X_list[k].dot(w[k]) for k in range(K)])
    prod_sign = np.prod(np.sign(margins), axis=0)
    log_sum   = np.sum(np.log(np.abs(margins) + 1e-12), axis=0)
    product   = prod_sign * np.exp(np.clip(log_sum, -500, 500))
    return (product <= 0).astype(np.int8)

def train_xor_lr(X_train, y_train, K, lam=1e-4, max_iter=1000, w0=None, seed=42):
    feat_size = X_train.shape[1]
    X_sp      = sparse.csr_matrix(X_train)
    X_list    = [X_sp for _ in range(K)]
    if w0 is None:
        np.random.seed(seed)
        w0 = np.random.randn(K * feat_size) * 0.01
    result = minimize(
        fun=xor_lr_loss_and_grad, x0=w0,
        args=(X_list, y_train.astype(float), K, feat_size, lam),
        jac=True, method='L-BFGS-B',
        options={'maxiter': max_iter, 'ftol': 1e-10, 'gtol': 1e-6, 'disp': False}
    )
    return result.x, feat_size

# ──────────────────────────────────────────────────────────────────────────────
#  5.  ONE-SHOT LINEAR APPROXIMATION ATTACK CORE
# ──────────────────────────────────────────────────────────────────────────────

def linear_approximation_attack(upper_w, lower_pufs, chal_size, n_levels, K_UP, K_d,
                                n_train, n_test, lam=1e-4, max_iter=1000):

    print(f"\n{'='*60}")
    print(f"  PC-LPUF Linear Approximation Attack (All-Zeros)")
    print(f"  chal_size={chal_size}, K_UP={K_UP}, K_d={K_d}")
    print(f"  CRPs : train={n_train:,}  test={n_test:,}")
    print(f"{'='*60}")

    # ── Generate CRPs ─────────────────────────────────────────────────────────
    print("\n[1] Generating CRPs...")
    np.random.seed(60)
    challenges_tr = np.random.choice([-1, 1], size=(n_train, chal_size))
    pos_tr        = np.random.randint(0, chal_size, n_train)
    val_tr        = np.random.randint(0, n_levels,  n_train)
    challenges_te = np.random.choice([-1, 1], size=(n_test,  chal_size))
    pos_te        = np.random.randint(0, chal_size, n_test)
    val_te        = np.random.randint(0, n_levels,  n_test)

    y_train = pclpuf_evaluate(upper_w, lower_pufs, challenges_tr, pos_tr, val_tr)
    y_test  = pclpuf_evaluate(upper_w, lower_pufs, challenges_te, pos_te, val_te)
    print(f"    Response balance: {y_train.mean()*100:.1f}% ones")

    # ── Linear Approximation Step (All Upper Bits = 0) ────────────────────────
    print("\n[2] Approximating Obfuscation Layer (Forcing Upper Bits = 0)...")
    guessed_upper_tr = np.zeros((K_UP, n_train), dtype=np.int8)
    guessed_upper_te = np.zeros((K_UP, n_test), dtype=np.int8)

    pos_tr_approx, val_tr_approx = xor_obfuscate_position_value(pos_tr, val_tr, guessed_upper_tr)
    pos_te_approx, val_te_approx = xor_obfuscate_position_value(pos_te, val_te, guessed_upper_te)

    # ── Construct Sparse Feature Matrices ─────────────────────────────────────
    print("[3] Engineering sparse linear feature maps...")
    Phi_tr = transform(challenges_tr)
    Phi_te = transform(challenges_te)

    X_train_approx = prepare_lr_features(Phi_tr, pos_tr_approx, val_tr_approx, chal_size, n_levels)
    X_test_approx  = prepare_lr_features(Phi_te, pos_te_approx, val_te_approx, chal_size, n_levels)

    # ── One-Shot Optimization using your exact train_xor_lr ───────────────────
    print("[4] Fitting multi-stream lower layer weights via L-BFGS-B...")
    w_down, feat_size_down = train_xor_lr(
        X_train_approx, y_train, K=K_d, lam=lam, max_iter=max_iter, w0=None, seed=42
    )

    # ── Evaluate Final Test Accuracy ──────────────────────────────────────────
    print("[5] Evaluating metrics on test dataset...")
    X_list_te = [sparse.csr_matrix(X_test_approx) for _ in range(K_d)]
    y_pred = predict_xor_lr(w_down, X_list_te, K_d, feat_size_down)

    final_accuracy = accuracy_score(y_test, y_pred)
    final_accuracy = max(final_accuracy, 1.0 - final_accuracy)

    print(f"\n{'─'*60}")
    print(f"  Attack Complete.")
    print(f"  Linearized XOR-{K_d} Model Test Accuracy: {final_accuracy * 100:.2f}%")
    print(f"{'─'*60}\n")

# ──────────────────────────────────────────────────────────────────────────────
#  6.  PIPELINE EXECUTION ENTRYPOINT
# ──────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Main experiment settings
    # chal_size: input challenge length for each PUF instance
    # n_levels: number of possible value states in the REAP-NVM obfuscation logic
    # K_UP: number of upper-layer APUF components
    # K_d: number of lower-layer REAP-NVM components used in the XOR structure
    # n_train: number of training CRPs used for the attack
    # n_test: number of test CRPs used for evaluation
    chal_size = 128
    n_levels  = 4
    K_UP      = 1
    K_d       = 3
    n_train   = 200_000
    n_test    = 40_000

    np.random.seed(60)
    upper_w    = apuf_generate(K_UP, chal_size, seed=0)
    lower_pufs = [ReapNVM(chal_size, seed=42 + k) for k in range(K_d)]

    linear_approximation_attack(
        upper_w    = upper_w,
        lower_pufs = lower_pufs,
        chal_size  = chal_size,
        n_levels   = n_levels,
        K_UP       = K_UP,
        K_d        = K_d,
        n_train    = n_train,
        n_test     = n_test,
        lam        = 1e-4,
        max_iter   = 1000
    )


In [ ]:
# XOR PUF Logistic Regression Attack
#
# This cell implements a baseline XOR-PUF attack using a product-form
# logistic regression loss. It is meant to verify the optimization recipe
# on a simpler XOR PUF before applying the same idea to the PC-LPUF case.

import numpy as np
from sklearn.metrics import accuracy_score
from scipy import sparse
from scipy.optimize import minimize
from scipy.special import expit

# ─── APUF ─────────────────────────────────────────────────────────────────────
def apuf_generate(k, chal_size, seed=0):
    """Generate K independent APUF weight vectors."""
    rng = np.random.default_rng(seed)
    return rng.normal(0, 1, (k, chal_size + 1))

def apuf_response(w, Phi):
    """Binary response of a single APUF using a linear threshold."""
    return (Phi @ w <= 0).astype(np.int8)

# ─── Parity Transform ─────────────────────────────────────────────────────────
def transform(challenges):
    """Build the parity feature matrix for APUF-style linear models."""
    N = challenges.shape[0]
    Phi_stages = np.cumprod(challenges, axis=1)
    bias = np.ones((N, 1))
    return np.hstack([Phi_stages, bias])

# ─── XOR PUF Evaluate ─────────────────────────────────────────────────────────
def xorpuf_evaluate(weights, challenges):
    """
    Evaluate an XOR-PUF by combining K APUF responses through a product of signs.
    The sign product is equivalent to XOR in the binary domain.
    """
    Phi = transform(challenges)
    # margins shape (K, N)
    margins = weights @ Phi.T  # (K, N)
    # sign of product = XOR in {-1,+1}
    product_sign = np.prod(np.sign(margins), axis=0)  # (N,)
    # convert to {0,1}: +1 -> 0, -1 -> 1
    return (product_sign <= 0).astype(np.int8)

# ─── Feature Builder (just parity vector for APUF) ───────────────────────────
def prepare_apuf_features(Phi):
    """Wrap the parity matrix as a sparse matrix for optimization."""
    return sparse.csr_matrix(Phi)

# ─── XOR LR Loss: product of margins then sigmoid ────────────────────────────
def xor_lr_loss_and_grad(w_flat, X_list, y, K, feat_size, lam=1e-4):
    """
    Logistic regression loss for an XOR of K linear models.
    Each stream produces a margin; the XOR decision is modeled by the
    product of the K margins, and the loss is computed via sigmoid.
    """
    N  = X_list[0].shape[0]
    w  = w_flat.reshape(K, feat_size)

    margins = np.array([X_list[k].dot(w[k]) for k in range(K)])  # (K, N)
    product = np.prod(margins, axis=0)                             # (N,)

    p = np.clip(expit(product), 1e-12, 1 - 1e-12)

    loss  = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
    loss += lam * 0.5 * np.dot(w_flat, w_flat)

    err  = (p - y) / N
    grad = np.zeros_like(w)
    for k in range(K):
        others   = np.prod(margins[np.arange(K) != k], axis=0)
        grad[k]  = X_list[k].T.dot(err * others) + lam * w[k]

    return loss, grad.flatten()

def predict_xor_lr(w_flat, X_list, K, feat_size):
    """Predict labels from the trained XOR logistic regression model."""
    w       = w_flat.reshape(K, feat_size)
    margins = np.array([X_list[k].dot(w[k]) for k in range(K)])
    product = np.prod(margins, axis=0)
    return (product <= 0).astype(np.int8)

# ─── Main ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Main experiment settings
    # chal_size: number of challenge bits in each APUF component
    # K: number of APUF components combined in the XOR PUF
    # n_train: number of training CRPs used for fitting the model
    # n_test: number of test CRPs used for evaluation
    chal_size = 128
    K         = 4
    n_train   = 400_000
    n_test    = 40_000

    np.random.seed(60)

    # ── Generate XOR PUF ──────────────────────────────────────────────────────
    weights = apuf_generate(K, chal_size, seed=0)

    print(f"XOR PUF (K={K}) — LR Attack with XOR product loss")

    # ── Generate CRPs ─────────────────────────────────────────────────────────
    print("Generating CRPs...")
    challenges_tr = np.random.choice([-1, 1], size=(n_train, chal_size))
    challenges_te = np.random.choice([-1, 1], size=(n_test,  chal_size))

    y_train = xorpuf_evaluate(weights, challenges_tr)
    y_test  = xorpuf_evaluate(weights, challenges_te)

    print(f"Response balance: {y_train.mean()*100:.1f}% ones")

    # ── Build features ────────────────────────────────────────────────────────
    print("Building features...")
    Phi_tr = transform(challenges_tr)
    Phi_te = transform(challenges_te)

    # Each XOR instance uses the same parity feature vector
    X_train_list = [prepare_apuf_features(Phi_tr) for _ in range(K)]
    X_test_list  = [prepare_apuf_features(Phi_te)  for _ in range(K)]

    feat_size = X_train_list[0].shape[1]
    print(f"Feature size per instance: {feat_size}")

    # ── Optimize XOR LR loss ──────────────────────────────────────────────────
    print("Optimizing XOR LR loss...")
    w0 = np.random.randn(K * feat_size) * 0.01

    result = minimize(
        fun=xor_lr_loss_and_grad,
        x0=w0,
        args=(X_train_list, y_train.astype(float), K, feat_size),
        jac=True,
        method='L-BFGS-B',
        options={'maxiter': 1000, 'ftol': 1e-10, 'gtol': 1e-6, 'disp': True}
    )

    w_opt = result.x

    # ── Evaluate ──────────────────────────────────────────────────────────────
    y_pred   = predict_xor_lr(w_opt, X_test_list, K, feat_size)
    accuracy = accuracy_score(y_test, y_pred)
    accuracy = max(accuracy, 1 - accuracy)
    print(f"\nFinal Accuracy XOR PUF (K={K}): {accuracy * 100:.2f}%")
    print(f"Expected: >95% since XOR PUF has no obfuscation")
